# LSVM Thesis Evaluation (Grouped Split, No Leakage)

This notebook follows the same workflow style as `FINAL_ML_V7-Copy1.ipynb`, but focuses on a thesis-ready evaluation of **Linear SVM** for win prediction.

## Objectives
- Use grouped 70/30 train-test split (`MATCH_KEY`) to avoid match-level leakage.
- Evaluate LSVM using:
  - Test Accuracy
  - ROC-AUC
  - Confusion Matrix
  - Classification Report (Precision, Recall, F1 for LOSS/WIN)
- Run 5-fold grouped cross-validation (ROC-AUC) for generalization.
- Compare LSVM vs Logistic Regression vs Decision Tree.


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

try:
    from IPython.display import HTML, display
except ImportError:
    # Fallback when running outside notebook/IPython environment
    def HTML(x):
        return x
    def display(x):
        print(x)

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier


## Dataset Setup

Using `original_files_ml/game_data_cleaned.csv` as evaluation/test dataset for the thesis experiment.

In [ ]:
# 1) Load test dataset for model evaluation
TEST_DATA_PATH = Path("original_files_ml/game_data_cleaned.csv")

if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {TEST_DATA_PATH.resolve()}")

df = pd.read_csv(TEST_DATA_PATH)

df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.upper()
      .str.replace(" ", "_", regex=False)
)

if "MATCH_KEY" not in df.columns:
    df["MATCH_KEY"] = df["EVENT_NAME"].astype(str) + "-" + df["MATCH_ID"].astype(str)

if "ROUND" in df.columns:
    df["ROUND"] = pd.to_numeric(df["ROUND"], errors="coerce")

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)

print("Evaluation dataset:", TEST_DATA_PATH.resolve())
print("Rows:", len(df))
print("Columns:", len(df.columns))


In [ ]:
# 2) Feature engineering (same win-feature logic)
df["ROUND_OFFENSE"] = (
    df.get("NUM_HAND_STRIKE", 0)
    + df.get("NUM_FOOT_STRIKE", 0)
    + df.get("NUM_DROPING_SCORE", 0)
)

df["CUM_OFFENSE"] = df.groupby(["MATCH_KEY", "CORNER"])["ROUND_OFFENSE"].cumsum()
df["CUM_VIOLATION"] = df.groupby(["MATCH_KEY", "CORNER"])["TOTAL_RAW_VIOLATION_COUNT"].cumsum()

# Attach final match result to each row
labels = df[["MATCH_KEY", "CORNER", "WIN_STATUS"]].copy()
labels["TARGET_WIN"] = labels["WIN_STATUS"].map({"WIN": 1, "LOSE": 0})
labels = labels.dropna(subset=["TARGET_WIN"]).drop_duplicates(["MATCH_KEY", "CORNER"], keep="last")

df = df.merge(labels[["MATCH_KEY", "CORNER", "TARGET_WIN"]], on=["MATCH_KEY", "CORNER"], how="left")

# Opponent cumulative features per round
opp = df[["MATCH_KEY", "ROUND", "CORNER", "CUM_OFFENSE", "CUM_VIOLATION"]].copy()
opp = opp.rename(columns={
    "CORNER": "CORNER_OPP",
    "CUM_OFFENSE": "CUM_OFFENSE_OPP",
    "CUM_VIOLATION": "CUM_VIOLATION_OPP",
})

df_win = df.merge(opp, on=["MATCH_KEY", "ROUND"], how="inner")
df_win = df_win[df_win["CORNER"] != df_win["CORNER_OPP"]].copy()

df_win["OFFENSE_ADV"] = df_win["CUM_OFFENSE"] - df_win["CUM_OFFENSE_OPP"]
df_win["VIOLATION_ADV"] = df_win["CUM_VIOLATION_OPP"] - df_win["CUM_VIOLATION"]

df_win = df_win.dropna(subset=["TARGET_WIN"]).copy()

FEATURES = ["CUM_OFFENSE", "CUM_VIOLATION", "OFFENSE_ADV", "VIOLATION_ADV", "ROUND"]

X = df_win[FEATURES].replace([np.inf, -np.inf], 0).fillna(0)
y = df_win["TARGET_WIN"].astype(int)
groups = df_win["MATCH_KEY"].astype(str)

print("Model rows:", len(df_win))
print("Unique matches:", df_win["MATCH_KEY"].nunique())
print("Class distribution:")
print(y.value_counts().sort_index())


## Train?Test Split (Grouped Split)

In [ ]:
# 3) Grouped train-test split (70/30) to prevent match leakage
RANDOM_STATE = 42

gss = GroupShuffleSplit(test_size=0.30, n_splits=1, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print(f"Train rows: {len(train_idx)} | Test rows: {len(test_idx)}")
print(f"Train matches: {groups_train.nunique()} | Test matches: {groups_test.nunique()}")


In [ ]:
# 4) Define models for comparison
models = {
    "LSVM (Calibrated)": CalibratedClassifierCV(
        estimator=Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        method="sigmoid",   # Platt scaling
        cv=5,
    ),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

models


## Algorithm Comparison Testing

In [ ]:
# 5) Evaluate all models on grouped split + grouped 5-fold CV ROC-AUC
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

comparison_rows = []
fitted_models = {}

for name, model in models.items():
    m = clone(model)
    m.fit(X_train, y_train)
    fitted_models[name] = m

    y_pred_test = m.predict(X_test)

    if hasattr(m, "predict_proba"):
        y_prob_train = m.predict_proba(X_train)[:, 1]
        y_prob_test = m.predict_proba(X_test)[:, 1]
    else:
        # Fallback for decision_function-only models
        train_scores = m.decision_function(X_train)
        test_scores = m.decision_function(X_test)
        y_prob_train = 1.0 / (1.0 + np.exp(-train_scores))
        y_prob_test = 1.0 / (1.0 + np.exp(-test_scores))

    train_auc = roc_auc_score(y_train, y_prob_train)
    test_auc = roc_auc_score(y_test, y_prob_test)
    test_acc = accuracy_score(y_test, y_pred_test)

    cv_scores = cross_val_score(
        clone(model),
        X,
        y,
        groups=groups,
        cv=sgkf,
        scoring="roc_auc",
        n_jobs=1,  # stable in restricted Windows env
    )

    comparison_rows.append({
        "Model": name,
        "Test_Accuracy": test_acc,
        "Test_ROC_AUC": test_auc,
        "Train_ROC_AUC": train_auc,
        "CV_ROC_AUC_Mean": cv_scores.mean(),
        "CV_ROC_AUC_STD": cv_scores.std(ddof=1),
        "CV_ROC_AUC_Scores": np.round(cv_scores, 4).tolist(),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("Test_ROC_AUC", ascending=False)
comparison_df


## Test Set Evaluation

In [ ]:
test_set_eval_df = comparison_df[["Model", "Test_Accuracy", "Test_ROC_AUC"]].copy()
test_set_eval_df = test_set_eval_df.sort_values("Test_ROC_AUC", ascending=False)
test_set_eval_df


## ROC?AUC Evaluation

In [ ]:
roc_auc_eval_df = comparison_df[["Model", "Train_ROC_AUC", "Test_ROC_AUC"]].copy()
roc_auc_eval_df = roc_auc_eval_df.sort_values("Test_ROC_AUC", ascending=False)
roc_auc_eval_df


## 5-Fold Cross-Validation

In [ ]:
cv_eval_df = comparison_df[["Model", "CV_ROC_AUC_Mean", "CV_ROC_AUC_STD", "CV_ROC_AUC_Scores"]].copy()
cv_eval_df = cv_eval_df.sort_values("CV_ROC_AUC_Mean", ascending=False)
cv_eval_df


## Classification Report (Precision, Recall, F1-Score)

In [ ]:
# LSVM class-wise metrics
lsvm = fitted_models["LSVM (Calibrated)"]

lsvm_pred = lsvm.predict(X_test)
lsvm_prob = lsvm.predict_proba(X_test)[:, 1]

lsvm_test_accuracy = accuracy_score(y_test, lsvm_pred)
lsvm_test_auc = roc_auc_score(y_test, lsvm_prob)
lsvm_cm = confusion_matrix(y_test, lsvm_pred, labels=[0, 1])

lsvm_report_df = pd.DataFrame(
    classification_report(
        y_test,
        lsvm_pred,
        labels=[0, 1],
        target_names=["LOSS", "WIN"],
        output_dict=True,
        zero_division=0,
    )
).T

print(f"LSVM Test Accuracy: {lsvm_test_accuracy:.4f}")
print(f"LSVM Test ROC-AUC:  {lsvm_test_auc:.4f}")

lsvm_report_df


## Confusion Matrix

In [ ]:
cm_df = pd.DataFrame(
    lsvm_cm,
    index=["Actual LOSS (0)", "Actual WIN (1)"],
    columns=["Pred LOSS (0)", "Pred WIN (1)"],
)
cm_df


## Overfitting Check / Generalization Testing

In [ ]:
# 7) Simple overfitting/generalization check
summary = comparison_df[["Model", "Train_ROC_AUC", "Test_ROC_AUC", "CV_ROC_AUC_Mean", "CV_ROC_AUC_STD"]].copy()
summary["Train_minus_CV"] = summary["Train_ROC_AUC"] - summary["CV_ROC_AUC_Mean"]
summary["Test_minus_CV"] = summary["Test_ROC_AUC"] - summary["CV_ROC_AUC_Mean"]
summary.sort_values("CV_ROC_AUC_Mean", ascending=False)


In [ ]:
# 8) Thesis-style HTML summary (like FINAL_ML notebook)
lsvm_row = comparison_df.loc[comparison_df["Model"] == "LSVM (Calibrated)"].iloc[0]
lr_row = comparison_df.loc[comparison_df["Model"] == "Logistic Regression"].iloc[0]
dt_row = comparison_df.loc[comparison_df["Model"] == "Decision Tree"].iloc[0]

summary_html = f"""
<h2 style='color:#1f3fb3; margin-bottom:8px;'>LSVM Thesis Evaluation Summary</h2>
<p style='margin-top:0;'>Dataset used for testing: <b>{TEST_DATA_PATH}</b></p>
<table style='border-collapse:collapse; width:100%; font-family:Arial, sans-serif;'>
  <thead>
    <tr style='background:#f2f6ff;'>
      <th style='border:1px solid #d0dbf7; padding:8px;'>Model</th>
      <th style='border:1px solid #d0dbf7; padding:8px;'>Test Accuracy</th>
      <th style='border:1px solid #d0dbf7; padding:8px;'>Test ROC-AUC</th>
      <th style='border:1px solid #d0dbf7; padding:8px;'>CV ROC-AUC (Mean &plusmn; SD)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style='border:1px solid #d0dbf7; padding:8px;'>LSVM (Calibrated)</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{lsvm_row['Test_Accuracy']:.4f}</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{lsvm_row['Test_ROC_AUC']:.4f}</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{lsvm_row['CV_ROC_AUC_Mean']:.4f} &plusmn; {lsvm_row['CV_ROC_AUC_STD']:.4f}</td>
    </tr>
    <tr>
      <td style='border:1px solid #d0dbf7; padding:8px;'>Logistic Regression</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{lr_row['Test_Accuracy']:.4f}</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{lr_row['Test_ROC_AUC']:.4f}</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{lr_row['CV_ROC_AUC_Mean']:.4f} &plusmn; {lr_row['CV_ROC_AUC_STD']:.4f}</td>
    </tr>
    <tr>
      <td style='border:1px solid #d0dbf7; padding:8px;'>Decision Tree</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{dt_row['Test_Accuracy']:.4f}</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{dt_row['Test_ROC_AUC']:.4f}</td>
      <td style='border:1px solid #d0dbf7; padding:8px;'>{dt_row['CV_ROC_AUC_Mean']:.4f} &plusmn; {dt_row['CV_ROC_AUC_STD']:.4f}</td>
    </tr>
  </tbody>
</table>
"""

display(HTML(summary_html))


## Interpretation Guide

Use the table outputs above to write your thesis interpretation:
- **Model quality on unseen test data:** compare `Test_Accuracy` and `Test_ROC_AUC`.
- **Generalization:** prioritize `CV_ROC_AUC_Mean` and lower `CV_ROC_AUC_STD`.
- **Overfitting signal:** large positive `Train_minus_CV` indicates possible overfitting.
- **Expected pattern in this dataset:** LSVM and Logistic Regression are usually close, while an unconstrained Decision Tree often overfits.
